# This Notebook estimates the model

## Settings

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from scipy.optimize import minimize

import DynamicTimeAllocationModel

# c++ settings
do_compile = True
threads = 64

import os
os.environ.pop('NoDefaultCurrentDirectoryInExePath', None)

do_reinstall_nlopt = False # If problems with NLOPT during re-compilation of c++ files, then try re-installing NLOPT by swithcing this to True and delete the folder "nlopt-2.4.2-dll64" in the cppfuncs folder before running this notebook.
if do_reinstall_nlopt:
    from EconModel import cpptools
    cpptools.setup_nlopt(folder='cppfuncs/', do_print=False,download=False,unzip=True)

In [ ]:
# setup model
settings = { 
       # technical settings
       'threads':threads,
       'do_multistart': False,
       'do_egm': True,
       'interp_method': 'linear',
       'interp_inverse': True,
       'precompute_intratemporal': True,
       'centered_gradient': True,
       'bargaining': 'limited',
}


model = DynamicTimeAllocationModel.HouseholdModelClass(par=settings) 
model.link_to_cpp(force_compile=do_compile)

## Empirical Moments to Match

In [ ]:
# all moments listed here will be used in estimation. Comment out those you do not want to use.
datamoms = dict()

# wages
datamoms['wage_level_w_25_34'] = 40.1
datamoms['wage_level_w_35_44'] = 49.3
datamoms['wage_level_m_25_34'] = 50.3
datamoms['wage_level_m_35_44'] = 67.8

# employment rates
datamoms['employment_rate_w_35_44'] = 64.0
datamoms['employment_rate_m_35_44'] = 88.0
datamoms['work_hours_w'] = 4.41*365 / 52.0  # daily hours to weekly hours
datamoms['work_hours_m'] = 5.7*365 / 52.0  # daily hours to weekly hours

# # consumption
datamoms['consumption'] = 42.716
datamoms['consumption_90_10_ratio'] = 3.33 * 1.0954

# # marriage and divorce rates
datamoms['marriage_rate_35_44'] = 69.0

# Mazzocco moments
datamoms['home_prod_w'] = (2.23+0.75+1.47+0.08) * 365/ 52
datamoms['home_prod_m'] = (1.6+0.54+0.88+0.1) * 365/ 52


# weights
weights = dict()
for mom in ('consumption_90_10_ratio',):
    weights[mom] = 10.0
    

## Parameters to estimate

In [ ]:
# parameters to estimate
estpars = {
    # Wages
    'mu': {'guess':2.3678,'lower':0.1,'upper':3.00}, 
    'mu_mult': {'guess':1.1126,'lower':1.0,'upper':3.0},
    'gamma': {'guess':0.1237,'lower':0.001,'upper':0.50},
    'gamma_mult': {'guess':1.7611,'lower':1.0,'upper':3.0},
    'sigma_mu': {'guess':0.5613,'lower':0.001,'upper':1.0},
    
    # Disutility from work
    'eta': {'guess':0.9033,'lower':0.1,'upper':5.0},
    'eta_mult': {'guess':0.8877,'lower':0.3,'upper':3.0},
    'phi': {'guess':4.4732,'lower':0.1,'upper':5.0},
    'phi_mult': {'guess':1.0855,'lower':0.3,'upper':3.0},
    
    # Home production
    'alpha': {'guess':0.9608,'lower':0.1,'upper':1.9},
    'pi': {'guess':0.6144,'lower':0.1,'upper':0.9},
    'lambda_': {'guess':5.7527,'lower':0.1,'upper':30.0},
    
    # # Match quality
    'sigma_love': {'guess':3.7895,'lower':0.01,'upper':20.5},
}

## setup initial guess 

In [ ]:
# check bounds
bounds_ok = True
for key in estpars.keys():
    if estpars[key]['guess']<estpars[key]['lower']:
        print(key,' lower',estpars[key]['guess'])
        bounds_ok = False
    
    if estpars[key]['guess']>estpars[key]['upper']:
        print(key,' upper',estpars[key]['guess'])
        bounds_ok = False

if not bounds_ok:
    stop

In [ ]:
# check initial guess
theta_init = np.array([estpars[key]['guess'] for key in estpars.keys()])
obj_init = model.obj_func(theta_init, estpars, datamoms, weights, do_print=True)

## Estimate model

In [ ]:
# Estimate model using nelder-mead algorithm
do_print = True
res = minimize(model.obj_func, theta_init, args=(estpars, datamoms,weights,do_print), method='Nelder-Mead',
               options={'xatol': 1e-3, 'fatol': 1e-3, 'disp': True, 'maxiter':500, 'maxfev':500})


In [ ]:
model.save_par('calibrated_par')